In [94]:
import pandas as pd
import numpy as np
# 读取CSV文件
df_apple = pd.read_csv("688981_posts_data.csv")


In [96]:
# 示例：假设所有时间都是 2023 年的
df_apple['Date'] = "2025-" + df_apple['Date']
df_apple['Date'] = pd.to_datetime(df_apple['Date'], format="%Y-%m-%d %H:%M")
df_apple.head()

,read_count,title,Date
0,5951,科创板公司攻坚“硬卡替” 带领集成电路等产业向高质量攀登,2025-04-13 05:40:00
1,1,，谁都不跟。就是压制,2025-04-14 01:48:00
2,320,我一直有一种预感，特朗普的半导体关税有可能是触发中芯国际大牛行情的关键因素,2025-04-14 12:36:00
3,43,大盘行情那么好，我手拿五个绿的，的，这么玩吗，飞要让老子销户,2025-04-14 10:43:00
4,19,美国关税反复，对中芯应该利好，为什么跌呢？,2025-04-14 01:44:00


In [80]:
import pandas as pd
import re



# 简单清洗文本（去除特殊符号、空格等）
def clean_text(text):
    text = re.sub(r'[^\w\s]', '', str(text))  # 去除非字母数字字符
    text = text.strip()  # 去除首尾空格
    return text

df_apple["title"] = df_apple["title"].apply(clean_text)

print(df_apple.head())  # 检查清洗后的数据

   read_count                                 title         update_time
0        5951            科创板公司攻坚硬卡替 带领集成电路等产业向高质量攀登 2025-04-13 05:40:00
1           1                              谁都不跟就是压制 2025-04-14 01:48:00
2         320  我一直有一种预感特朗普的半导体关税有可能是触发中芯国际大牛行情的关键因素 2025-04-14 12:36:00
3          43            大盘行情那么好我手拿五个绿的的这么玩吗飞要让老子销户 2025-04-14 10:43:00
4          19                    美国关税反复对中芯应该利好为什么跌呢 2025-04-14 01:44:00


In [98]:
from transformers import AutoModel, AutoTokenizer
from transformers import pipeline, AutoModelForSequenceClassification

# Model used: https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest?text=Covid+cases+are+increasing+fast%21
# Load tokenizer and model from Hugging Face Hub
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True)
# for getting embeddings, it is important to have output_hidden_states=True, otherwise, you wont get embeddings

# Create a sentiment analysis pipeline using the loaded model and tokenizer
sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# Label_2: Positive
# Label_1: Neutral
# Label_0: Negative 
results_apple = [(tweet, sentiment_pipeline(tweet)) for tweet in df_apple["title"]]
results_apple


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


[('科创板公司攻坚“硬卡替” 带领集成电路等产业向高质量攀登',
  [{'label': 'neutral', 'score': 0.8386176824569702}]),
 ('，谁都不跟。就是压制', [{'label': 'neutral', 'score': 0.7380134463310242}]),
 ('我一直有一种预感，特朗普的半导体关税有可能是触发中芯国际大牛行情的关键因素',
  [{'label': 'neutral', 'score': 0.8132067918777466}]),
 ('大盘行情那么好，我手拿五个绿的，的，这么玩吗，飞要让老子销户',
  [{'label': 'neutral', 'score': 0.810854434967041}]),
 ('美国关税反复，对中芯应该利好，为什么跌呢？',
  [{'label': 'neutral', 'score': 0.8219490051269531}]),
 ('整个A股就一个板块是绿的', [{'label': 'neutral', 'score': 0.7728021144866943}]),
 ('要做好突破千元的思想准备！', [{'label': 'neutral', 'score': 0.767923891544342}]),
 ('这股专坑国产代替情怀大怨种', [{'label': 'neutral', 'score': 0.7645836472511292}]),
 ('还不抢只能说脑壳里少了东西的。', [{'label': 'neutral', 'score': 0.8033933639526367}]),
 ('中心国际弱的不行了', [{'label': 'neutral', 'score': 0.7370137572288513}]),
 ('明天要不就收绿，要不就高开低走，一定不能再低开高走了。一切都是为了技术指标回调，',
  [{'label': 'neutral', 'score': 0.8022447228431702}]),
 ('反弹看来差不多了，开盘又是全天最高点吗？', [{'label': 'neutral', 'score': 0.8022174835205078}]),
 ('自主可控+内销为主+技术护城河越发重要',

In [100]:

final_scores = []
for result in results_apple:
    sentiment = result[1][0]
    label = sentiment["label"]
    score = sentiment["score"]

    # LABEL_0: Negative
    # LABEL_1: Neutral
    # LABEL_2: Positive
    if label == "positive":
        final_scores.append(score)
    elif label == "negative":
        final_scores.append(-1*score)
    else:
        final_scores.append(0)
df_apple.loc[:,"score"] = final_scores
df_apple


,read_count,title,Date,score
0,5951,科创板公司攻坚“硬卡替” 带领集成电路等产业向高质量攀登,2025-04-13 05:40:00,0.0
1,1,，谁都不跟。就是压制,2025-04-14 01:48:00,0.0
2,320,我一直有一种预感，特朗普的半导体关税有可能是触发中芯国际大牛行情的关键因素,2025-04-14 12:36:00,0.0
3,43,大盘行情那么好，我手拿五个绿的，的，这么玩吗，飞要让老子销户,2025-04-14 10:43:00,0.0
4,19,美国关税反复，对中芯应该利好，为什么跌呢？,2025-04-14 01:44:00,0.0
...,...,...,...,...
496,47,"国产替代,中国自主",2025-04-10 09:18:00,0.0
497,48,今天要大涨了,2025-04-10 09:14:00,0.0
498,118,加关税半导体设备赢一次，海外建厂苹果设备铲子股再赢一次，应该是赢两次吧,2025-04-10 08:58:00,0.0
499,60,这货不应该跌得这么惨。没骨气。,2025-04-10 08:48:00,0.0


In [102]:
df_apple.describe()

,read_count,Date,score
count,501.000000,501,501.000000
mean,688.746507,2025-04-10 19:04:17.365269504,0.015837
min,1.000000,2025-02-09 08:00:00,0.000000
25%,75.000000,2025-04-10 10:06:00,0.000000
50%,139.000000,2025-04-11 05:18:00,0.000000
75%,274.000000,2025-04-12 11:43:00,0.000000
max,36887.000000,2025-04-14 12:45:00,0.917571
std,2975.364898,NaN,0.112867


In [104]:
import yfinance as yf

# 中芯国际的A股代码是 688981.SS（上海证券交易所）
ticker = "688981.SS"  # 或者 "688981.SS"（A股） / "0981.HK"（港股）

# 获取历史数据
data = yf.download(
    ticker,
    start="2025-03-01",  # 开始日期
    end="2025-04-15",    # 结束日期
    progress=False       # 不显示进度条
)

print(data.head())  # 查看前5行

Price           Close        High        Low       Open    Volume
Ticker      688981.SS   688981.SS  688981.SS  688981.SS 688981.SS
Date                                                             
2025-03-03  96.000000   98.099998  95.180000  96.989998  49878742
2025-03-04  97.279999   99.070000  94.610001  95.000000  44469412
2025-03-05  97.570000   98.199997  96.160004  97.300003  35097289
2025-03-06  98.660004  100.300003  98.370003  98.849998  56756728
2025-03-07  96.879997   98.550003  96.519997  97.599998  39398650


In [112]:
# 安装seaborn可视化库（如果未安装）
# %pip install seaborn
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# 将两个数据框的日期列统一转换为标准日期格式（去除时间部分）
df_apple['Date'] = pd.to_datetime(df_apple['Date']).dt.normalize()
aapl_stock['Date'] = pd.to_datetime(aapl_stock['Date']).dt.normalize()

# 检查两个数据集的日期范围是否匹配
print("股票数据日期范围:", aapl_stock['Date'].min(), "至", aapl_stock['Date'].max())
print("情感分析数据日期范围:", df_apple['Date'].min(), "至", df_apple['Date'].max())

# 筛选情感数据，使其日期范围与股票数据保持一致
df_apple_filtered = df_apple[
    (df_apple['Date'] >= aapl_stock['Date'].min()) & 
    (df_apple['Date'] <= aapl_stock['Date'].max())
]

# 按日期聚合情感分数（取每日均值）
df_apple_agg = df_apple_filtered.groupby('Date').agg({'score': 'mean'}).reset_index()

# 将处理后的情感数据与股票数据按日期合并（内连接）
df_merged = pd.merge(aapl_stock, df_apple_agg, on='Date', how='inner')

# 检查合并后的数据行数
print("合并后的数据总行数:", len(df_merged))

# 创建自定义颜色映射：红色(负面)->灰色(中性)->绿色(正面)
colors = [(1, 0, 0), (0.5, 0.5, 0.5), (0, 1, 0)]  # RGB格式
n_bins = 100  # 将颜色分为100个渐变区间
cmap_name = 'sentiment_cmap'  # 颜色映射名称
sentiment_cmap = LinearSegmentedColormap.from_list(cmap_name, colors, N=n_bins)

# 将情感分数标准化到[0,1]区间用于颜色映射
norm = plt.Normalize(df_merged['score'].min(), df_merged['score'].max())
sm = plt.cm.ScalarMappable(cmap=sentiment_cmap, norm=norm)
df_merged['color'] = df_merged['score'].apply(lambda x: sm.to_rgba(x))

# 绘制双轴图表
fig, ax1 = plt.subplots(figsize=(14, 7))

# 主坐标轴：绘制股票收盘价曲线（蓝色）
sns.lineplot(x='Date', y='Close', data=df_merged, ax=ax1, color='blue', label='苹果股价')
ax1.set_ylabel('股价', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# 副坐标轴：绘制情感分数散点图（颜色映射）
ax2 = ax1.twinx()
ax2.scatter(df_merged['Date'], df_merged['score'], color=df_merged['color'], label='情感分数')
ax2.axhline(y=0.0, color='black', linestyle='--', linewidth=0.5)  # 添加中性参考线
ax2.set_ylabel('情感分数', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# 添加标题和图例
plt.title('股价与情感分数趋势对比')
fig.tight_layout()  # 自动调整子图间距
ax1.legend(loc='upper left')  # 主图例在左上
ax2.legend(loc='upper right')  # 副图例在右上

plt.show()

KeyError: 'Date'